In [1]:
import numpy as np

# 替换为你的文件路径
file_path = '../data/course/course_rqvae_codes.npy'

# 加载数据
data = np.load(file_path, allow_pickle=True)

# 查看基本信息
print(data[:10])
print(f"数据类型: {type(data)}")
print(f"数组维度 (Shape): {data.shape}")
print(f"元素类型 (Dtype): {data.dtype}")

[[6 1 2 0]
 [2 6 2 0]
 [2 6 4 0]
 [2 6 4 1]
 [2 6 4 2]
 [4 6 1 0]
 [4 5 6 0]
 [2 6 0 0]
 [7 6 2 0]
 [0 0 3 0]]
数据类型: <class 'numpy.ndarray'>
数组维度 (Shape): (707, 4)
元素类型 (Dtype): int32


In [2]:
import h5py
rec_data = []
with h5py.File('../data/user_item_interact.h5', 'r') as f1:
    user_ids = f1['user_id'][:]          # 提取所有 user_id
    user_profile = f1['user_profile'].asstr(encoding='utf-8')[:] # 提取所有 user_profile
    item_lists = f1['item_id_list'][:]    # 提取所有变长数组
    rec_data = list(zip(user_ids, user_profile, item_lists))
# print(rec_data[:5])
print(len(item_lists))

199199


In [ ]:
CODEBOOK_SIZE = 8

def item_to_offset_code(item_id):
    """Convert item_id (1-indexed) to offset semantic code, consistent with item2code() in data_vision.py."""
    raw_code = data[item_id - 1]  # data loaded from cell 0, 0-indexed
    return [int(c + i * CODEBOOK_SIZE + 1) for i, c in enumerate(raw_code)]

train_list = []
test_list = []

for uid, user_seq in zip(user_ids, item_lists):
    seq_len = len(user_seq)
    
    # 至少有2个行为
    if seq_len < 2:
        continue
    
    # 逻辑分支 
    if seq_len == 2:
        # 只有两个数据 [A, B]，不进入测试集，直接把这唯一的映射 [A] -> B 喂给训练
        train_list.append({
            'user_id': int(uid),
            'history': [item_to_offset_code(user_seq[0])],
            'target': item_to_offset_code(user_seq[1])
        })
    else:
        # 情况 B：长度 >= 3
        # 测试集划分 (用前 n-1 预测第 n 个)
        test_list.append({
            'user_id': int(uid),
            'history': [item_to_offset_code(iid) for iid in user_seq[:-1]],
            'target': item_to_offset_code(user_seq[-1])
        })
        
        # 训练集滑动(因为是双向注意力) (为了不泄露，训练集 target 最高只到倒数第二个物品)
        end_idx = seq_len - 2 
        for i in range(1, end_idx + 1):
            train_list.append({
                'user_id': int(uid),
                'history': [item_to_offset_code(iid) for iid in user_seq[:i]],
                'target': item_to_offset_code(user_seq[i])
            })

print(f"训练样本数: {len(train_list)}")
print(f"测试样本数: {len(test_list)}")

训练样本数: 388131
测试样本数: 95423


In [4]:
import h5py
import numpy as np

def save_single_h5(file_name, data_list):
    """
    将包含 {'user_id': int, 'history': [[c0,c1,c2,c3],...], 'target': [c0,c1,c2,c3]} 的列表保存为单个 H5 文件
    """
    with h5py.File(file_name, 'w') as f:
        uids = np.array([item['user_id'] for item in data_list], dtype='int32')
        f.create_dataset("user_id", data=uids)
        
        targets = np.array([item['target'] for item in data_list], dtype='int32')
        f.create_dataset("target", data=targets)
        
        vlen_int = h5py.special_dtype(vlen=np.dtype('int32'))
        ds_hist = f.create_dataset("history", (len(data_list),), dtype=vlen_int)
        for i, item in enumerate(data_list):
            ds_hist[i] = np.array(item['history'], dtype='int32').flatten()
            
    print(f"成功保存至: {file_name} | 样本数: {len(data_list)}")

save_single_h5("../data/tiger/train_dataset.h5", train_list)
save_single_h5("../data/tiger/test_dataset.h5", test_list)

成功保存至: ../data/tiger/train_dataset.h5 | 样本数: 388131
成功保存至: ../data/tiger/test_dataset.h5 | 样本数: 95423


In [5]:
with h5py.File('../data/tiger/train_dataset.h5', 'r') as f:
        # user_ids = f['user_id'][:]          # 提取所有 user_id
        # user_profile = f['user_profile'].asstr(encoding='utf-8')[:] # 提取所有 user_profile
        item_history = f['history'][:]
        item_target = f['target'][:]
        # print(item_history[:10])
        # print(item_target[:108])
        print(item_history[:10])
        print(item_target[:10])


[array([ 3, 15, 17, 26], dtype=int32)
 array([ 3, 15, 17, 26,  3, 16, 21, 25], dtype=int32)
 array([ 3, 15, 17, 26,  3, 16, 21, 25,  4, 16, 17, 25], dtype=int32)
 array([ 5, 15, 24, 33], dtype=int32) array([ 3, 15, 17, 25], dtype=int32)
 array([ 5, 15, 19, 26], dtype=int32)
 array([ 5, 15, 19, 26,  1,  9, 20, 29], dtype=int32)
 array([ 5, 15, 19, 26,  1,  9, 20, 29,  4, 16, 17, 25], dtype=int32)
 array([ 5, 15, 19, 26,  1,  9, 20, 29,  4, 16, 17, 25,  3, 16, 21, 25],
       dtype=int32)
 array([ 5, 15, 19, 26,  1,  9, 20, 29,  4, 16, 17, 25,  3, 16, 21, 25,  3,
        15, 18, 29], dtype=int32)                                          ]
[[ 3 16 21 25]
 [ 4 16 17 25]
 [ 3 15 19 29]
 [ 3 15 18 29]
 [ 7 12 22 25]
 [ 1  9 20 29]
 [ 4 16 17 25]
 [ 3 16 21 25]
 [ 3 15 18 29]
 [ 3 15 24 25]]
